# Kafka streaming ingestion (bronze → silver → gold)

Deployed by the `kafka_streaming_ingestion` DAB and run as a **continuous job task**, so the three streaming queries restart automatically on failure. All environment differences (catalog, schema, Kafka cluster, secret scope, checkpoint root) arrive as **job parameters** — this notebook is byte-for-byte identical in dev and prod.

* **Bronze** — raw Kafka messages byte-for-byte + full metadata, append-only, durable checkpoint
* **Silver** — explicit-schema parsing; malformed payloads quarantined to a dead-letter queue
* **Gold** — exactly-once MERGE upserts into a latest-state table

In [0]:
# ============================================================
# PARAMETERS - injected by the job task (resources/kafka_streaming_ingestion.job.yml)
# Defaults below only apply when running this notebook interactively.
# ============================================================
dbutils.widgets.text("catalog", "main", "Target catalog")
dbutils.widgets.text("schema", "dbdemos_data_ingestion", "Target schema")
dbutils.widgets.text("kafka_bootstrap_servers", "broker1:9092,broker2:9092", "Kafka bootstrap servers")
dbutils.widgets.text("kafka_topic", "events", "Kafka topic")
dbutils.widgets.text("kafka_secret_scope", "kafka", "Secret scope holding Kafka user/pwd")
dbutils.widgets.text("checkpoint_root", "/Volumes/main/dbdemos_data_ingestion/raw_data", "Checkpoint root (UC volume)")

catalog                 = dbutils.widgets.get("catalog")
schema                  = dbutils.widgets.get("schema")
KAFKA_BOOTSTRAP_SERVERS = dbutils.widgets.get("kafka_bootstrap_servers")
KAFKA_TOPIC             = dbutils.widgets.get("kafka_topic")
KAFKA_SECRET_SCOPE      = dbutils.widgets.get("kafka_secret_scope")
CHECKPOINT_ROOT         = dbutils.widgets.get("checkpoint_root")

# --- Unity Catalog targets ---
bronze_table = f"{catalog}.{schema}.kafka_bronze_events"
silver_table = f"{catalog}.{schema}.kafka_silver_events"
dlq_table    = f"{catalog}.{schema}.kafka_events_dlq"       # dead-letter queue for bad payloads
gold_table   = f"{catalog}.{schema}.kafka_events_latest"    # upserted latest state

# --- Checkpoints: one DEDICATED location per streaming query ---
checkpoint_bronze = f"{CHECKPOINT_ROOT}/checkpoints/kafka/bronze"
checkpoint_silver = f"{CHECKPOINT_ROOT}/checkpoints/kafka/silver"
checkpoint_gold   = f"{CHECKPOINT_ROOT}/checkpoints/kafka/gold"

# --- Kafka auth via secrets - credentials NEVER hardcoded ---
KAFKA_SECURITY_OPTIONS = {
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism":    "SCRAM-SHA-512",
    "kafka.sasl.jaas.config": (
        "org.apache.kafka.common.security.scram.ScramLoginModule required "
        f'username="{dbutils.secrets.get(KAFKA_SECRET_SCOPE, "user")}" '
        f'password="{dbutils.secrets.get(KAFKA_SECRET_SCOPE, "pwd")}";'
    ),
}

print(f"source : {KAFKA_BOOTSTRAP_SERVERS} / topic '{KAFKA_TOPIC}'")
print(f"bronze : {bronze_table}")
print(f"silver : {silver_table}  (DLQ -> {dlq_table})")
print(f"gold   : {gold_table}")
print(f"ckpt   : {CHECKPOINT_ROOT}")

In [0]:
# ======================================================================
# BRONZE - land raw Kafka messages byte-for-byte, keeping ALL metadata
# ======================================================================
#   * maxOffsetsPerTrigger : backpressure - caps what each micro-batch pulls
#   * minPartitions        : forces parallelism even with few Kafka partitions
#   * startingOffsets       : FIRST start only; afterwards the durable
#                            checkpoint resumes exactly where the query stopped

from pyspark.sql import functions as F

raw_kafka = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 100_000)
    .option("minPartitions", 8)
    .options(**KAFKA_SECURITY_OPTIONS)
    .load()
)

bronze_query = (
    raw_kafka.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_bronze)
    .queryName("kafka_bronze_ingest")
    .trigger(processingTime="30 seconds")
    .toTable(bronze_table)   # creates the table on first run, appends afterwards
)

print(f"Bronze query {bronze_query.id} streaming raw messages into {bronze_table}")

In [0]:
# ======================================================================
# SILVER - parse payloads with an EXPLICIT schema, quarantine bad records
# ======================================================================
# Never infer schemas on streaming data: one malformed message must not break
# the pipeline. from_json() returns NULL on invalid payloads, and those rows are
# quarantined into a dead-letter queue (DLQ) table for reprocessing.

EVENT_SCHEMA = """
  event_id BIGINT,
  user_id STRING,
  source_system STRING,
  amount DOUBLE,
  event_time TIMESTAMP
"""

def parse_and_quarantine(batch_df, batch_id):
    """Split each micro-batch into valid rows (silver) and bad payloads (DLQ)."""
    decoded = (
        batch_df
        .selectExpr(
            "topic", "partition", "offset", "timestamp",
            "CAST(key   AS STRING) AS key",
            "CAST(value AS STRING) AS value",
        )
        .withColumns({"parsed": F.from_json("value", EVENT_SCHEMA)})
    )
    valid = decoded.filter("parsed IS NOT NULL").select(
        "topic", "partition", "offset", "timestamp", "key", "parsed.*"
    )
    dead = decoded.filter("parsed IS NULL").select(
        "topic", "partition", "offset", "timestamp", "key", "value"
    )

    # append mode creates the table on first use; replaying a failed batch
    # is guarded downstream by the gold MERGE (idempotent)
    valid.write.mode("append").saveAsTable(silver_table)
    if not dead.isEmpty():
        dead.write.mode("append").saveAsTable(dlq_table)
        print(f"batch {batch_id}: {dead.count()} malformed payload(s) routed to {dlq_table}")

silver_query = (
    spark.readStream.table(bronze_table)
    .writeStream
    .foreachBatch(parse_and_quarantine)
    .option("checkpointLocation", checkpoint_silver)
    .queryName("kafka_silver_parse")
    .trigger(processingTime="30 seconds")
    .start()
)

print(f"Silver query {silver_query.id} parsing {bronze_table} -> {silver_table}")

In [0]:
# ======================================================================
# GOLD - exactly-once MERGE upserts into the latest-state table
# ======================================================================
# foreachBatch + MERGE is idempotent: replaying a failed batch changes nothing.

from delta.tables import DeltaTable

# MERGE requires the target table to exist
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {gold_table} (
    event_id BIGINT, user_id STRING, source_system STRING,
    amount DOUBLE, event_time TIMESTAMP, topic STRING,
    partition INT, `offset` BIGINT, `timestamp` TIMESTAMP
  )
""")

def upsert_latest_state(batch_df, batch_id):
    deduped = batch_df.dropDuplicates(["event_id"])   # guard against producer replays
    target = DeltaTable.forName(spark, gold_table)
    (
        target.alias("t")
        .merge(deduped.alias("s"), "t.event_id = s.event_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

gold_query = (
    spark.readStream.table(silver_table)
    .writeStream
    .foreachBatch(upsert_latest_state)
    .option("checkpointLocation", checkpoint_gold)
    .queryName("kafka_gold_upsert")
    .trigger(processingTime="1 minute")
    .start()
)

print(f"Gold query {gold_query.id} upserting {silver_table} -> {gold_table}")

## Operating in production

**Monitoring** — the continuous job keeps the streams alive and restarts them on failure.

* `for q in spark.streams.active: print(q.name, q.status, q.lastProgress["batchId"])` — batch id, input/processing rates, batch duration
* If `inputRowsPerSecond` persistently exceeds `processedRowsPerSecond`, scale out (more `minPartitions` or bigger compute); `maxOffsetsPerTrigger` keeps each batch committable
* The Kafka consumer-group lag is the source-of-truth delay metric

**Failure & restart semantics** — durable checkpoints + atomic Delta commits mean no data loss and no duplicates on restart. Never delete checkpoint folders unless you deliberately want to re-read from `startingOffsets`.

**Backfills** — `.trigger(availableNow=True)` processes everything available then stops (scheduled catch-up); `startingTimestamp` or per-partition JSON `startingOffsets` for point-in-time replays.

**Environment isolation** — dev and prod deploy this same notebook; only target variables differ (catalog, schema, secret scope, checkpoint root, Kafka cluster). State (checkpoints, tables, secrets) is never shared across environments.

**Graceful shutdown** — `for q in spark.streams.active: q.stop()`